# 스포츠 손상 환자의 회복 기간 예측 모델 (1단계: 실제 데이터셋)

**과제:** 첨단융합약학 - 약물과 스포츠의 상관관계

**목표:** 부상 종류, 치료법(Recovery_Therapy_Type), 생체지표, 생활습관 등을 입력값으로 하여 회복 기간(Recovery_Time, 회귀) 및 회복 성공 여부(Recovery_Success, 분류)를 예측하는 모델을 구축한다.

**데이터셋:** `Athlete_recovery_dataset.csv` (1,000행 × 17열, 결측치 없음)

## 1. 라이브러리 및 데이터 로드

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                              accuracy_score, classification_report,
                              confusion_matrix, ConfusionMatrixDisplay)

df = pd.read_csv('Athlete_recovery_dataset.csv')
print(df.info())
df.head()

## 2. 데이터 전처리

In [ ]:
# 2-1. Blood_Pressure ("120/80") -> Systolic / Diastolic 분리
bp_split = df['Blood_Pressure'].str.split('/', expand=True)
df['Systolic_BP'] = bp_split[0].astype(int)
df['Diastolic_BP'] = bp_split[1].astype(int)
df = df.drop(columns=['Blood_Pressure'])

# 2-2. 순서형(Ordinal) 변수 인코딩 (Mild < Moderate < Severe)
ordinal_map = {'Mild': 0, 'Moderate': 1, 'Severe': 2}
for col in ['Injury_Severity', 'Muscle_Recovery_Status', 'Imaging_Report_Severity']:
    df[col] = df[col].map(ordinal_map)

intensity_map = {'Low': 0, 'Medium': 1, 'High': 2}
df['Training_Intensity'] = df['Training_Intensity'].map(intensity_map)

# 2-3. 명목형(Nominal) 변수 -> One-Hot Encoding
df = pd.get_dummies(df, columns=['Injury_Type', 'Recovery_Therapy_Type'], drop_first=True)

# 2-4. 불필요한 ID 컬럼 제거
df = df.drop(columns=['Athlete_ID'])

print('전처리 후 컬럼:', df.columns.tolist())
df.head()

## 3. 탐색적 데이터 분석 (EDA)

In [ ]:
# 3-1. Recovery_Time 분포
plt.figure(figsize=(6, 4))
sns.histplot(df['Recovery_Time'], kde=True, bins=20)
plt.title('Distribution of Recovery Time')
plt.xlabel('Recovery Time (days)')
plt.tight_layout()
plt.show()

In [ ]:
# 3-2. 상관관계 히트맵
corr_cols = ['Recovery_Time', 'Heart_Rate', 'Systolic_BP', 'Diastolic_BP',
              'POMS_Score', 'Confidence_Score', 'Sleep_Hours', 'Dietary_Intake',
              'Training_Days_per_Week', 'Recovery_Days_per_Week',
              'Injury_Severity', 'Training_Intensity']

plt.figure(figsize=(10, 8))
sns.heatmap(df[corr_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()

# Recovery_Time과 다른 변수 간 상관계수 절댓값 확인
print(df[corr_cols].corr()['Recovery_Time'].sort_values(key=abs, ascending=False))

**해석:** Recovery_Time과 다른 모든 변수 간 상관계수의 절댓값이 0.1 미만으로 매우 낮다. 이는 본 데이터셋의 변수들이 회복 기간을 설명할 수 있는 통계적 신호를 거의 포함하지 않음을 의미한다.

## 4. Feature/Target 분리 및 데이터 분할

In [ ]:
y = df['Recovery_Time']
X = df.drop(columns=['Recovery_Time', 'Recovery_Success'])  # Recovery_Success는 결과 변수이므로 제외 (data leakage 방지)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('Train:', X_train.shape, ' Test:', X_test.shape)

## 5. 회귀 모델링: Recovery_Time 예측

In [ ]:
results = {}

# 5-1. Linear Regression (베이스라인)
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
results['Linear Regression'] = lr.predict(X_test_scaled)

# 5-2. Random Forest Regressor
rf = RandomForestRegressor(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)
results['Random Forest'] = rf.predict(X_test)

print('모델 학습 완료')

## 6. 회귀 모델 평가

In [ ]:
eval_table = []
for name, y_pred in results.items():
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    eval_table.append({'Model': name, 'MAE': mae, 'RMSE': rmse, 'R2': r2})
    print(f'{name:18s} | MAE: {mae:.3f} | RMSE: {rmse:.3f} | R2: {r2:.3f}')

pd.DataFrame(eval_table)

In [ ]:
# 실제값 vs 예측값 산점도 (Random Forest)
plt.figure(figsize=(6, 6))
plt.scatter(y_test, results['Random Forest'], alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('Actual Recovery Time')
plt.ylabel('Predicted Recovery Time')
plt.title('Random Forest: Actual vs Predicted')
plt.tight_layout()
plt.show()

**해석:** R²가 음수(-0.04 ~ -0.05)로 나타나, 모델이 단순히 평균값을 예측하는 것보다도 성능이 낮다. 즉, 현재 변수들로는 회복 기간을 예측할 수 없으며, 이는 3절의 상관관계 분석 결과와 일치한다.

## 7. Feature Importance 분석 (회귀)

In [ ]:
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print(importances.head(10))

plt.figure(figsize=(8, 6))
importances.head(10).sort_values().plot(kind='barh')
plt.title('Top 10 Feature Importances (Random Forest Regressor)')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

## 8. [추가] Recovery_Success 분류 모델

In [ ]:
# Recovery_Time과 Recovery_Success는 강한 상관관계가 있으므로 둘 다 입력값에서 제외
y_clf = df['Recovery_Success']
X_clf = df.drop(columns=['Recovery_Time', 'Recovery_Success'])

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)

scaler_c = StandardScaler()
X_train_c_scaled = scaler_c.fit_transform(X_train_c)
X_test_c_scaled = scaler_c.transform(X_test_c)

clf_results = {}

# Logistic Regression
logit = LogisticRegression(max_iter=1000, random_state=42)
logit.fit(X_train_c_scaled, y_train_c)
clf_results['Logistic Regression'] = logit.predict(X_test_c_scaled)

# Random Forest Classifier
rfc = RandomForestClassifier(n_estimators=200, random_state=42)
rfc.fit(X_train_c, y_train_c)
clf_results['Random Forest Classifier'] = rfc.predict(X_test_c)

for name, y_pred_c in clf_results.items():
    acc = accuracy_score(y_test_c, y_pred_c)
    print(f'\n--- {name} (Accuracy: {acc:.3f}) ---')
    print(classification_report(y_test_c, y_pred_c, zero_division=0))

In [ ]:
# Confusion Matrix (Random Forest Classifier)
cm = confusion_matrix(y_test_c, clf_results['Random Forest Classifier'])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Fail(0)', 'Success(1)'])
disp.plot(cmap='Blues')
plt.title('Confusion Matrix - Random Forest Classifier')
plt.tight_layout()
plt.show()

**해석:** Accuracy는 약 67%이나, 클래스 불균형(실패 679 vs 성공 321)으로 인해 모델이 대부분 '실패(0)'로 예측하는 경향을 보인다. 성공(1) 클래스의 Recall이 0에 가까워, 실질적인 분류 성능은 낮다.

## 9. 결론 (1단계)

- 전체 머신러닝 파이프라인(전처리, EDA, 회귀/분류 모델링, 평가, Feature Importance)을 완성하였다.
- 회귀 모델의 R²가 음수, 분류 모델의 Recall이 0에 가까운 것으로 보아, 본 데이터셋은 변수 간 신호가 거의 없는(synthetic/random) 데이터로 판단된다.
- 이에 따라 2단계 노트북(`02_synthetic_drug_recovery_analysis.ipynb`)에서는 약리학적 가정을 반영한 가상 데이터셋을 생성하여, 동일한 파이프라인으로 약물(NSAIDs/스테로이드) 처방과 회복 기간의 관계를 재검증한다.